# Mapowanie sprawozdań na standard biznesradar## Co wgrać na początek| plik | rola ||---|---|| `brmap.py` | cała logika || `start.py` | program interaktywny || `slownik.xlsx` | **jedyny plik z wiedzą** — rośnie z każdą spółką || sprawozdanie spółki `.xlsx` | dane wejściowe || plik wzorcowy `TICKER.xlsx` | opcjonalny, do kontroli poprawności |Dwa sposoby pracy:- **Punkt 2 — program interaktywny.** Jedna komórka, pyta o wszystko po kolei. Najprostsze.- **Punkty 3–6 — krok po kroku.** Gdy chcesz kontrolować każdy etap albo coś debugować.

## 1. KonfiguracjaWybierz `TRYB`:| tryb | kiedy | słownik po zamknięciu sesji ||---|---|---|| `"github"` | **zalecany** — repo klonuje się **na Dysk**, więc `git pull` / `git push` działa i nic nie ginie | zostaje || `"dysk"` | pliki wgrane ręcznie na Dysk, bez gita | zostaje || `"upload"` | szybki jednorazowy test | **ginie** |> Uwaga: gdyby sklonować repo do `/content`, wszystkie zmiany w `slownik.xlsx`> przepadłyby razem z sesją. Dlatego tryb `"github"` klonuje do folderu na Dysku.

In [ ]:
TRYB = "github"     # "github" | "dysk" | "upload"REPO = "https://github.com/TWOJ/biznesradar-mapowanie.git"   # <- podmienBAZA = "/content/drive/MyDrive"                              # folder na DyskuKATALOG = "biznesradar-mapowanie"                            # nazwa folderu repoimport os, sys, subprocesstry:    import openpyxlexcept ImportError:    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl"], check=True)if TRYB in ("github", "dysk"):    from google.colab import drive    drive.mount("/content/drive")if TRYB == "github":    cel = os.path.join(BAZA, KATALOG)    if os.path.isdir(os.path.join(cel, ".git")):        os.chdir(cel)        print("Repo juz jest na Dysku - pobieram zmiany:")        print(subprocess.run(["git", "pull"], capture_output=True, text=True).stdout)    else:        os.makedirs(BAZA, exist_ok=True)        os.chdir(BAZA)        subprocess.run(["git", "clone", REPO, KATALOG], check=True)        os.chdir(cel)elif TRYB == "dysk":    os.makedirs(os.path.join(BAZA, KATALOG), exist_ok=True)    os.chdir(os.path.join(BAZA, KATALOG))else:    os.chdir("/content")    print("Wgraj: brmap.py, start.py, slownik.xlsx oraz plik sprawozdania")    from google.colab import files    files.upload()sys.path.insert(0, os.getcwd())print("\nFolder roboczy:", os.getcwd())print("Pliki:", sorted(f for f in os.listdir(".") if f.endswith((".py", ".xlsx"))))

In [ ]:
if not os.path.exists("brmap.py"):    raise SystemExit("Brak brmap.py w folderze roboczym - sprawdz TRYB i sciezki w komorce wyzej.")import importlib, brmap, startimportlib.reload(brmap); importlib.reload(start)print("Gotowe.")

## 2. Program interaktywny (najprostsza droga)Zadaje pytania po kolei: skrót spółki, plik sprawozdania, plik wzorcowy. Nieznaną spółkęobsługuje kreatorem i **czeka**, aż uzupełnisz słownik. Efektem jest `wynik_TICKER.xlsx`.> W Colabie uruchamiaj **tylko tak, jak niżej**. `!python start.py` nie zadziała —> podproces nie ma dostępu do klawiatury i program przerwie się z komunikatem.

In [ ]:
import startstart.main()      # folder roboczy ustawiła komórka konfiguracyjna

## 3. Nowa spółka — kreator (krok po kroku)Uruchom **tylko przy pierwszym** sprawozdaniu danej spółki. Przy kolejnych kwartałach przejdź od razu do punktu 3.Kreator wykrywa m.in. czy dane są w tysiącach czy milionach oraz czy sumy bilansowe **otwierają** bloki (jak u Neuki), czy je **zamykają** (jak u KGHM).

In [ ]:
PLIK   = "raport_spolki.xlsx"     # <- plik sprawozdaniaTICKER = "ABC"                     # <- skrot spolkiNAZWA  = "Nazwa Spolki"brmap.nowa_spolka(PLIK, "slownik.xlsx", TICKER, NAZWA)

### Co zrobić z plikiem `propozycja_<TICKER>.xlsx`Otwórz go i przejdź dwa arkusze:**`1_Wiersz_do_Spolki`** — skopiuj ten wiersz do arkusza `Spolki` w `slownik.xlsx`. Sprawdź `mnoznik` (1 = tysiące, 1000 = miliony).**`3_Propozycje_aliasow`** — popraw kolumnę `standard_key`, potem skopiuj kolumny **A–I** do arkusza `Aliasy`. Kolumny J, K, L są tylko do przeglądu.| kolor | co znaczy ||---|---|| zielony | pewne, można zatwierdzić || żółty | prawdopodobne, sprawdź || pomarańczowy | słabe dopasowanie — podpowiedź, nie odpowiedź || czerwony | brak podpowiedzi, zmapuj ręcznie |Progi są ustawione tak, żeby w zielonym nie pojawiła się błędna podpowiedź — bo taka zostałaby zatwierdzona bez sprawdzenia.

## 4. MapowaniePo uzupełnieniu słownika. Powtarzaj aż `Niezmapowanych: 0`, a walidacja pokaże `14/14 OK`.

In [ ]:
# Jesli druga kolumna sprawozdania to POPRZEDNI KWARTAL (typowe dla MSSF) - zostaw None.# Jesli to ten sam kwartal ROK WCZESNIEJ (typowe dla ustawy o rachunkowosci),# dociagnij bilans poprzedniego kwartalu z pliku standardowego:#   PREV = brmap.wczytaj_okres_wzorca(f"{TICKER}.xlsx", "2025-12-31")PREV = Noneout, wyniki, walid, niezmapowane = brmap.mapuj("slownik.xlsx", TICKER, PLIK, bilans_poprzedni=PREV)

Wynik ma cztery arkusze:- **Wynik** — kolumna gotowa do wklejenia do standardu- **Walidacja** — 14 sum kontrolnych (aktywa = pasywa, EBIT, przepływy, ...)- **Audyt** — z którego wiersza raportu wzięła się każda liczba- **Niezmapowane** — co jeszcze zostało do zrobienia

## 5. Porównanie ze wzorcemJeśli spółka jest już wprowadzona do standardu — najszybszy sposób na wyłapanie błędów mapowania.

In [ ]:
# tolerancja 2 dla spolek raportujacych w ZLOTYCH (mnoznik 0.001) - zaokraglenia daja +/-1TOL = 2.0 if PLIK and brmap.zbadaj_plik(PLIK)["mnoznik"] < 1 else 1.0brmap.porownaj(f"{TICKER}.xlsx", "2026-03-31", out, tol=TOL)

## 6. Pobranie wynikówW trybie „dysk" pliki są już na Dysku — ta komórka jest potrzebna tylko przy trybie „upload”.

In [ ]:
from google.colab import filesfor f in (out, f"porownanie_{TICKER}.xlsx"):    if os.path.exists(f):        files.download(f)

---### Ściąga```pythonbrmap.zbadaj_plik(plik)                       # podglad wykrytego ukladu (arkusz, kolumny, mnoznik, sekcje)brmap.nowa_spolka(plik, slownik, ticker)      # kreator konfiguracji i propozycji aliasowbrmap.mapuj(slownik, ticker, plik)            # mapowanie + walidacjabrmap.porownaj(wzorzec, data, wynik)          # kontrola ze wzorcembrmap.wczytaj_okres_wzorca(wzorzec, data)     # bilans poprzedniego kwartalu ze standardu```**Dwa parametry, o ktorych latwo zapomniec:**`bilans_poprzedni` — gdy druga kolumna sprawozdania to ten sam kwartal rok wczesniej(uklad ustawy o rachunkowosci), kapitalu obrotowego nie da sie policzyc z samego pliku.Bez tego cztery pozycje przeplywow policza sie po cichu rok do roku.`tol=2` — dla spolek raportujacych w zlotych. Kazda pozycja jest zaokraglanado pelnych tysiecy, wiec sumy kontrolne potrafia rozjechac sie o 1.Jesli mapowanie sie nie zgadza, zacznij od arkusza **Audyt** — pokazuje sciezke kazdej liczbyod wiersza w raporcie do klucza standardu.

## 7. Odesłanie zmian słownika do repoDotyczy tylko trybu `"github"`. Po dopisaniu aliasów warto odesłać `slownik.xlsx` na GitHub,żeby zmiany nie zostały tylko na Twoim Dysku.GitHub nie przyjmuje zwykłego hasła — potrzebny jest **personal access token**(GitHub → Settings → Developer settings → Personal access tokens → uprawnienie `repo`).Wklej go w okienko, które pojawi się pod komórką; nie zapisuj go w kodzie notatnika.

In [ ]:
import getpass, subprocesssubprocess.run(["python", "slownik_csv.py"], check=True)      # czytelne diffysubprocess.run(["python", "test_slownik.py"], check=True)      # kontrola spojnosciuzytkownik = input("Nazwa uzytkownika GitHub: ").strip()token = getpass.getpass("Personal access token (nie bedzie widoczny): ").strip()opis = input("Opis zmiany: ").strip() or "Aktualizacja slownika"subprocess.run(["git", "config", "user.email", "mf@biznesradar.pl"], check=True)subprocess.run(["git", "config", "user.name", uzytkownik], check=True)subprocess.run(["git", "add", "slownik.xlsx", "slownik_csv"], check=True)subprocess.run(["git", "commit", "-m", opis], check=False)url = REPO.replace("https://", f"https://{uzytkownik}:{token}@")print(subprocess.run(["git", "push", url, "HEAD"], capture_output=True, text=True).stderr[-400:])subprocess.run(["git", "remote", "set-url", "origin", REPO], check=False)   # token nie zostaje w konfiguracjiprint("Gotowe.")